In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
# Generate questions for the llm-zoomcamp course
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
documents = documents_llm
len(documents_llm)

113

In [3]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()
openai_client = OpenAI()
user_prompt = json.dumps(doc)

In [7]:
# Create the messages
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [8]:
# Call the model
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)
result = response.output_parsed

print(result)
print("\n")
print(result.questions)

questions=['Can I still join the course if I found it late, or is it already too late to sign up?', 'If I join now, can I still get the certificate, or did I miss that chance?', 'What do I need to do to be eligible for the course certificate if I start after the course has begun?', 'Is it okay to join the course later, and will I still be able to submit the project for certification?', 'Do I need to submit my project before submissions close in order to get the certificate?']


['Can I still join the course if I found it late, or is it already too late to sign up?', 'If I join now, can I still get the certificate, or did I miss that chance?', 'What do I need to do to be eligible for the course certificate if I start after the course has begun?', 'Is it okay to join the course later, and will I still be able to submit the project for certification?', 'Do I need to submit my project before submissions close in order to get the certificate?']


In [9]:
from evaluation_utils import llm_structured

"""
llm_structured: calls the OpenAI API with structured output
llm_structured_retry: retries structured-output calls when a request fails
calc_price: calculates the price from token usage
calc_total_price: calculates the total price from multiple usage objects
map_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.
"""

"\nllm_structured: calls the OpenAI API with structured output\nllm_structured_retry: retries structured-output calls when a request fails\ncalc_price: calculates the price from token usage\ncalc_total_price: calculates the total price from multiple usage objects\nmap_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.\n"

In [10]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I discovered it late?', 'If I join now, is it still possible to get the certificate?', 'Do I need to submit my project before submissions close to receive a certificate?', 'Am I allowed to start the course after it already began?', 'What’s the deadline for submitting the project if I want the certificate?']


In [11]:
# Tracking cost
usage.input_tokens, usage.output_tokens

(207, 83)

In [12]:
from evaluation_utils import calc_price
cost = calc_price(usage)
cost

{'input_cost': 0.00015525,
 'output_cost': 0.00037349999999999997,
 'total_cost': 0.0005287499999999999}

In [13]:
# convert these questions into ground truth records:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I discovered it late?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, is it still possible to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to receive a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course after it already began?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for submitting the project if I want the certificate?',
  'document': '74eb249bbf'}]

Generating Ground Truth for All Documents

In [14]:
from evaluation_utils import llm_structured_retry

# Processing function
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [15]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

# Sequence processing
for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
# Parallel processing
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

ground_truth = []
usages = []

# Split the generated records and the token usage into separate lists
for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)


  0%|          | 0/113 [00:00<?, ?it/s]

565

In [17]:
from evaluation_utils import calc_price
from evaluation_utils import calc_total_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

print(total_cost)
print(calc_total_price(usages))

0.0882375
0.0882375


In [19]:
# Create a dataframe with ground truth records
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)